# Dataset columns (`DataSet/{symbol}.csv`)

Each row is one **trading session** in a calendar month: equity data from yfinance, ATM call/put from Polygon, Black–Scholes (European, no dividend yield) for IV and Greeks. Months are stacked in one file.

## Calendar & equity

| Column | Description |
|--------|-------------|
| **Date** | Session date. |
| **Stock_Close** | Underlying adjusted close (yfinance). |
| **Stock_Dividends** | Dividend paid that day (0 if none). |
| **r** | Annual risk-free rate as a decimal (Polygon 1-month Treasury, daily series aligned to `Date`). |
| **RV** | Annualized realized volatility of underlying returns: 20-day rolling stdev × √252 (computed on full history; may be NaN until enough history exists). |

## Option identifiers & prices

| Column | Description |
|--------|-------------|
| **Call_Sym** / **Put_Sym** | Polygon option tickers (`O:…` OCC format) for the ATM straddle for that month. |
| **Call_Close** / **Put_Close** | Daily closing prices. Missing Polygon prints are forward/back-filled within the month before Greeks; stale on gap days. |
| **Call_imp_vol** / **Put_imp_vol** | Implied volatility σ (decimal, e.g. 0.20 = 20%) per leg. From Polygon when present; else inverted from price with BS + `brentq`. |
| **Straddle_imp_vol** | Single σ such that BS call(σ)+put(σ) equals **Call_Close + Put_Close** (same `S`, `K`, `T`, `r`); bisection on [1e−4, 2]. Uses the `math` BS helpers. |

## Straddle Greeks (sum of legs)

Each **Straddle_**{name} = **Call_**{name} + **Put_**{name} (row-wise). Interpretation: total sensitivity of **long call + long put** when each leg uses its own BS σ (**Call_imp_vol** / **Put_imp_vol**). If the two IVs differ, this differs from Greeks of a single-σ model at **Straddle_imp_vol**.

| Column | Definition |
|--------|------------|
| **Straddle_Delta** | Call_Delta + Put_Delta |
| **Straddle_Gamma** | Call_Gamma + Put_Gamma |
| **Straddle_Vega** | Call_Vega + Put_Vega |
| **Straddle_Theta** | Call_Theta + Put_Theta |
| **Straddle_Rho** | Call_Rho + Put_Rho |
| **Straddle_Vanna** | Call_Vanna + Put_Vanna |
| **Straddle_Volga** | Call_Volga + Put_Volga |

## Per-leg Greeks (call vs put)

All are **Black–Scholes** for that leg’s own σ (**Call_*** uses **Call_imp_vol**, **Put_*** uses **Put_imp_vol**), with **K** and expiry from the symbols. Computed on the panel after quote alignment.

| Column | Description |
|--------|-------------|
| **Delta** | ∂V/∂S (call: N(d1); put: N(d1)−1). |
| **Gamma** | ∂²V/∂S² (same functional form for call and put at same inputs). |
| **Vega** | ∂V/∂σ as **raw** sensitivity: S·φ(d1)·√T (not scaled per 1 vol point). |
| **Theta** | Time decay, **per year** (standard BS Θ). |
| **Rho** | Sensitivity to the **risk-free rate**: approximate change in option value for a **+1 percentage point** move in r (e.g. 0.05 → 0.06), i.e. (∂V/∂r)/100. |
| **Vanna** | ∂²V/(∂S ∂σ) = ∂Δ/∂σ; implemented as φ(d1)·(√T − d1/σ). |
| **Volga** | ∂²V/∂σ² (“vomma”): vega_raw·d1·d2/σ. |

Prefix **Call_** or **Put_** selects the leg. Where inputs are invalid (NaN price/IV, non-positive T, etc.), Greek cells are NaN.

## Flags

| Column | Description |
|--------|-------------|
| **Force_Close** | `True` only on the **last** equity session row kept for that month (after dropping non-trading weekdays); `False` otherwise. Useful to tag month-end rolls. |


In [46]:
import os
import yfinance as yf
import pandas as pd
from scipy.optimize import brentq
from scipy.stats import norm
from datetime import datetime, timedelta,date
import requests
import json
from dotenv import load_dotenv
from polygon import RESTClient
from datetime import date as date_type, timedelta
import pandas as pd
import re
import math
import numpy as np

load_dotenv()
POLYGON_API_KEY = os.getenv("Polygon_API_Key")
if not POLYGON_API_KEY:
    raise ValueError("Set Polygon_API_Key in your .env file")


## Underlying

yfinance daily history for `symbol`: close, dividends, returns, 20-day annualized realized vol (`RV`) → `underlying_df` (Date index).

In [53]:
symbol = "ACMR"
ticker = yf.Ticker(symbol)
history = ticker.history(start="2023-12-01", actions=True, auto_adjust=False)
underlying_df = pd.DataFrame(history)
underlying_df.reset_index(inplace=True)
underlying_df["Date"] = underlying_df["Date"].dt.date
if "Dividends" not in underlying_df.columns:
    underlying_df["Dividends"] = 0.0
else:
    underlying_df["Dividends"] = underlying_df["Dividends"].fillna(0.0)
underlying_df["Return"] = underlying_df["Close"].pct_change()
underlying_df["RV"] = underlying_df["Return"].rolling(window=20).std() * np.sqrt(252)
underlying_df = underlying_df.set_index("Date").sort_index()


## Month starts

`month_starts`: 1st of each month from `start_date` to `end_date` (option selection days).

In [54]:
start_date = date(2024, 5, 1)
end_date = date(2026, 3, 31)

month_starts = []
y, m = start_date.year, start_date.month
while True:
    first = date(y, m, 1)
    if first > end_date:
        break
    if first >= start_date:
        month_starts.append(first)
    if m == 12:
        y, m = y + 1, 1
    else:
        m += 1

## Output

Equity lives in `underlying_df`; the notebook writes a single `DataSet/{symbol}.csv` (no separate underlying export).

## Risk-free rate

Polygon 1-month Treasury yield (daily, % → decimal), forward-filled; `r_fallback` if empty.

In [55]:
client = RESTClient(POLYGON_API_KEY)
from_date = min(month_starts).strftime("%Y-%m-%d")
to_date = date.today().strftime("%Y-%m-%d")
yields_raw = list(client.list_treasury_yields(date_gte=from_date, date_lte=to_date, limit=50000, sort="date", order="asc"))
r_fallback = 0.0365
if not yields_raw:
    r_by_date = pd.Series(r_fallback, index=pd.date_range(from_date, to_date, freq="D"))
    print("No Treasury yields from Polygon; using fallback rate for all dates.")
else:
    dates_list = []
    rates_list = []
    for i in range(len(yields_raw)):
        d = yields_raw[i].date
        y = yields_raw[i].yield_1_month
        dates_list.append(pd.to_datetime(d))
        rates_list.append((float(y) / 100.0) if y is not None else np.nan)
    r_by_date = pd.Series(rates_list, index=dates_list)
    r_by_date = r_by_date.sort_index()
    full_range = pd.date_range(from_date, to_date, freq="D")
    r_by_date = r_by_date.reindex(full_range).ffill().fillna(r_fallback)
    r_by_date = r_by_date.astype(np.float64)
    print(f"Loaded 1-month Treasury yields from {r_by_date.index.min()} to {r_by_date.index.max()} (n={len(r_by_date)})")

Loaded 1-month Treasury yields from 2024-05-01 00:00:00 to 2026-04-12 00:00:00 (n=712)


## ATM options

`atm_option`: next-month expiry (first Friday, else fallbacks in first week), nearest ATM strike with call+put (or best call/put + warning).

In [56]:
def atm_option(symbol, s0, date):
    client = RESTClient(POLYGON_API_KEY)

    date_obj = date
    date_str = date_obj.strftime("%Y-%m-%d")

    if date_obj.month == 12:
        next_month_first = date_obj.replace(year=date_obj.year + 1, month=1, day=1)
    else:
        next_month_first = date_obj.replace(month=date_obj.month + 1, day=1)

    ny, nm = next_month_first.year, next_month_first.month
    days_to_friday = (4 - next_month_first.weekday()) % 7
    first_friday = next_month_first + timedelta(days=days_to_friday)
    second_friday = first_friday + timedelta(weeks=1)
    exp_first = first_friday.strftime("%Y-%m-%d")
    exp_second = second_friday.strftime("%Y-%m-%d")

    contracts = list(
        client.list_options_contracts(underlying_ticker=symbol, as_of=date_str, limit=1000)
    )
    if not contracts:
        print(f"No options contracts found as of {date_str}.")
        return None, None

    def exp_in_first_week(exp_s: str) -> bool:
        ed = datetime.strptime(exp_s, "%Y-%m-%d").date()
        return ed.year == ny and ed.month == nm and 1 <= ed.day <= 7

    extra_exps = sorted(
        {c.expiration_date for c in contracts if c.expiration_date and exp_in_first_week(c.expiration_date)},
        key=lambda s: abs((datetime.strptime(s, "%Y-%m-%d").date() - first_friday).days),
    )
    expiry_order = []
    for e in [exp_first, exp_second] + extra_exps:
        if e not in expiry_order:
            expiry_order.append(e)

    def pick_pair(calls, puts, exp_used: str):
        common = sorted(
            {c.strike_price for c in calls} & {p.strike_price for p in puts},
            key=lambda k: abs(k - s0),
        )
        if common:
            k = common[0]
            atm_call = next(c for c in calls if c.strike_price == k)
            atm_put = next(p for p in puts if p.strike_price == k)
            if exp_used != exp_first:
                print(
                    f"ATM expiry fallback: using {exp_used} (no chain on primary {exp_first} as of {date_str})."
                )
            return atm_call.ticker, atm_put.ticker
        atm_call = min(calls, key=lambda c: abs(c.strike_price - s0))
        atm_put = min(puts, key=lambda p: abs(p.strike_price - s0))
        if exp_used != exp_first:
            print(
                f"ATM expiry fallback: using {exp_used} (no chain on primary {exp_first} as of {date_str})."
            )
        print(
            "Warning: no strike with both call and put; using best call / best put (strikes may differ)."
        )
        return atm_call.ticker, atm_put.ticker

    for exp in expiry_order:
        calls = [c for c in contracts if c.expiration_date == exp and c.contract_type == "call"]
        puts = [c for c in contracts if c.expiration_date == exp and c.contract_type == "put"]
        if not calls or not puts:
            continue
        return pick_pair(calls, puts, exp)

    print(
        f"No ATM pair found as of {date_str}; tried expiries (first week next month): {expiry_order}"
    )
    return None, None

## Options math

BS prices/IV (`brentq` when IV missing). Straddle IV: `math` BS + bisection vs `Call_Close+Put_Close`. Per-leg Greeks; `Straddle_*` Greeks = call + put. Panel: `ffill`/`bfill` option fields then recompute (Polygon vs equity calendar gaps).

In [57]:
def bs_call_price(S, K, T, r, sigma):
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)


def bs_put_price(S, K, T, r, sigma):
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)


IV_FALLBACK = 1.0


def implied_vol_call(market_price, S, K, T, r):
    def obj_func(sigma):
        return bs_call_price(S, K, T, r, sigma) - market_price

    try:
        return brentq(obj_func, 1e-6, 5)
    except ValueError:
        return IV_FALLBACK


def implied_vol_put(market_price, S, K, T, r):
    def obj_func(sigma):
        return bs_put_price(S, K, T, r, sigma) - market_price

    try:
        return brentq(obj_func, 1e-6, 5)
    except ValueError:
        return IV_FALLBACK


def norm_cdf_math(x):
    return 0.5 * (1 + math.erf(x / math.sqrt(2)))


def bs_call_price_math(S, K, T, r, sigma):
    if T <= 0:
        return max(S - K, 0)
    d1 = (math.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * math.sqrt(T))
    d2 = d1 - sigma * math.sqrt(T)
    return S * norm_cdf_math(d1) - K * math.exp(-r * T) * norm_cdf_math(d2)


def bs_put_price_math(S, K, T, r, sigma):
    if T <= 0:
        return max(K - S, 0)
    d1 = (math.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * math.sqrt(T))
    d2 = d1 - sigma * math.sqrt(T)
    return K * math.exp(-r * T) * norm_cdf_math(-d2) - S * norm_cdf_math(-d1)


def bs_straddle_price_math(S, K, T, r, sigma):
    return bs_call_price_math(S, K, T, r, sigma) + bs_put_price_math(S, K, T, r, sigma)


def straddle_implied_vol(C, P, S, K, T, r=0.0, tol=1e-6, max_iter=100):
    target = C + P
    low = 1e-4
    high = 2.0
    for _ in range(max_iter):
        mid = 0.5 * (low + high)
        price = bs_straddle_price_math(S, K, T, r, mid)
        if abs(price - target) < tol:
            return mid
        if price > target:
            high = mid
        else:
            low = mid
    return mid


def add_bs_greeks(df: pd.DataFrame, K: float, option_type: str) -> pd.DataFrame:
    S = df["S"].to_numpy(dtype=float)
    T = np.maximum(df["ttm"].to_numpy(dtype=float), 1e-12)
    r = df["r"].to_numpy(dtype=float)
    sig = np.maximum(df["imp_vol"].to_numpy(dtype=float), 1e-6)
    sqrtT = np.sqrt(T)
    d1 = (np.log(S / K) + (r + 0.5 * sig**2) * T) / (sig * sqrtT)
    d2 = d1 - sig * sqrtT
    pdf1 = norm.pdf(d1)

    gamma = pdf1 / (S * sig * sqrtT)
    vega_raw = S * pdf1 * sqrtT
    dd1_dsigma = sqrtT - d1 / sig
    vanna = pdf1 * dd1_dsigma
    volga = vega_raw * d1 * d2 / sig

    if option_type == "C":
        delta = norm.cdf(d1)
        theta_ann = -S * pdf1 * sig / (2 * sqrtT) - r * K * np.exp(-r * T) * norm.cdf(d2)
        rho_1pct = K * T * np.exp(-r * T) * norm.cdf(d2) / 100.0
    else:
        delta = norm.cdf(d1) - 1.0
        theta_ann = -S * pdf1 * sig / (2 * sqrtT) + r * K * np.exp(-r * T) * norm.cdf(-d2)
        rho_1pct = -K * T * np.exp(-r * T) * norm.cdf(-d2) / 100.0

    out = df.copy()
    out["delta"] = delta
    out["gamma"] = gamma
    out["vega"] = vega_raw
    out["theta"] = theta_ann
    out["rho"] = rho_1pct
    out["vanna"] = vanna
    out["volga"] = volga
    return out


def parse_occ_expiry_strike(option_ticker: str):
    symbol_part = option_ticker.split(":")[1]
    match = re.search(r"^[A-Z]+(\d{6})([CP])(\d{8})$", symbol_part)
    if not match:
        raise ValueError(f"Could not parse option symbol: {option_ticker}")
    expiry_str = match.group(1)
    if len(expiry_str) == 6:
        expiry_str = "20" + expiry_str
    expiry = pd.to_datetime(expiry_str, format="%Y%m%d").date()
    K = int(match.group(3)) / 1000.0
    return expiry, K


def vectorized_leg_greeks(S, K, T, r, sigma, is_call: bool):
    n = len(S)
    keys = ["Delta", "Gamma", "Vega", "Theta", "Rho", "Vanna", "Volga"]
    out = {k: np.full(n, np.nan, dtype=float) for k in keys}
    S = np.asarray(S, dtype=float)
    T = np.asarray(T, dtype=float)
    r = np.asarray(r, dtype=float)
    sigma = np.asarray(sigma, dtype=float)
    valid = (
        np.isfinite(S)
        & np.isfinite(T)
        & np.isfinite(r)
        & np.isfinite(sigma)
        & (T > 1e-10)
        & (sigma > 1e-8)
        & np.isfinite(K)
        & (K > 0)
        & (S > 0)
    )
    if not valid.any():
        return out
    S_v = S[valid]
    T_v = T[valid]
    r_v = r[valid]
    sig_v = np.maximum(sigma[valid], 1e-6)
    sqrtT = np.sqrt(T_v)
    d1 = (np.log(S_v / K) + (r_v + 0.5 * sig_v**2) * T_v) / (sig_v * sqrtT)
    d2 = d1 - sig_v * sqrtT
    pdf1 = norm.pdf(d1)
    gamma_v = pdf1 / (S_v * sig_v * sqrtT)
    vega_v = S_v * pdf1 * sqrtT
    dd1_dsigma = sqrtT - d1 / sig_v
    vanna_v = pdf1 * dd1_dsigma
    volga_v = vega_v * d1 * d2 / sig_v
    if is_call:
        delta_v = norm.cdf(d1)
        theta_v = -S_v * pdf1 * sig_v / (2 * sqrtT) - r_v * K * np.exp(-r_v * T_v) * norm.cdf(d2)
        rho_v = K * T_v * np.exp(-r_v * T_v) * norm.cdf(d2) / 100.0
    else:
        delta_v = norm.cdf(d1) - 1.0
        theta_v = -S_v * pdf1 * sig_v / (2 * sqrtT) + r_v * K * np.exp(-r_v * T_v) * norm.cdf(-d2)
        rho_v = -K * T_v * np.exp(-r_v * T_v) * norm.cdf(-d2) / 100.0
    out["Delta"][valid] = delta_v
    out["Gamma"][valid] = gamma_v
    out["Vega"][valid] = vega_v
    out["Theta"][valid] = theta_v
    out["Rho"][valid] = rho_v
    out["Vanna"][valid] = vanna_v
    out["Volga"][valid] = volga_v
    return out


_GREEK_COLS = ["delta", "gamma", "vega", "theta", "rho", "vanna", "volga"]
_OPTION_BAR_COLS = ["date", "close", "imp_vol", *_GREEK_COLS]


def option_bars_with_iv(
    option_ticker: str,
    d_start: date,
    d_end: date,
    r_by_date: pd.Series,
    underlying_df: pd.DataFrame,
    r_fallback: float,
) -> pd.DataFrame:
    client = RESTClient(POLYGON_API_KEY)
    from_str = d_start.strftime("%Y-%m-%d")
    to_str = d_end.strftime("%Y-%m-%d")
    aggs = client.get_aggs(
        ticker=option_ticker,
        multiplier=1,
        timespan="day",
        from_=from_str,
        to=to_str,
        limit=50000,
    )
    if not aggs:
        return pd.DataFrame(columns=_OPTION_BAR_COLS)

    rows = []
    for agg in aggs:
        rows.append(
            {
                "date": pd.to_datetime(agg.timestamp, unit="ms").date(),
                "close": agg.close,
                "imp_vol": getattr(agg, "implied_volatility", None) or getattr(agg, "iv", None),
            }
        )
    df = pd.DataFrame(rows).sort_values("date").reset_index(drop=True)

    symbol_part = option_ticker.split(":")[1]
    match = re.search(r"^[A-Z]+(\d{6})([CP])(\d{8})$", symbol_part)
    if not match:
        raise ValueError(
            f"Could not parse option symbol: {option_ticker}. Expected format like 'AAPL240209C00185000'."
        )

    expiry_str = match.group(1)
    option_type = match.group(2)
    strike_str = match.group(3)
    if len(expiry_str) == 6:
        expiry_str = "20" + expiry_str
    expiry = pd.to_datetime(expiry_str, format="%Y%m%d").date()
    K = int(strike_str) / 1000.0

    df["ttm"] = df["date"].apply(lambda x: max((expiry - x).days / 365.25, 1e-9))
    if isinstance(r_by_date, pd.Series):
        df["r"] = (
            r_by_date.reindex(pd.to_datetime(df["date"])).ffill().bfill().fillna(r_fallback).values
        )
    else:
        df["r"] = float(r_by_date)

    spot_df = underlying_df[["Close"]].reset_index()
    spot_df.columns = ["date", "S"]
    spot_df["date"] = pd.to_datetime(spot_df["date"]).dt.date
    df = df.merge(spot_df, on="date", how="left")
    last_S = float(underlying_df["Close"].iloc[-1]) if len(underlying_df) else np.nan
    df["S"] = df["S"].ffill().bfill().fillna(last_S)

    df["imp_vol"] = pd.to_numeric(df["imp_vol"], errors="coerce")
    need_iv = df["imp_vol"].isna() | ~np.isfinite(df["imp_vol"])
    if need_iv.any():
        if option_type == "C":
            df.loc[need_iv, "imp_vol"] = df.loc[need_iv].apply(
                lambda row: implied_vol_call(row["close"], row["S"], K, row["ttm"], row["r"]),
                axis=1,
            )
        else:
            df.loc[need_iv, "imp_vol"] = df.loc[need_iv].apply(
                lambda row: implied_vol_put(row["close"], row["S"], K, row["ttm"], row["r"]),
                axis=1,
            )
    df["imp_vol"] = df["imp_vol"].astype(float)
    df = add_bs_greeks(df, K, option_type)
    return df.drop(columns=["ttm", "r", "S"])

## Build CSV

Per month: business days → merge equity (drop no `Stock_Close`) → merge options → align quotes, per-leg Greeks, `Straddle_imp_vol`, summed `Straddle_{Greek}` → `Force_Close` on last row. Concat → `DataSet/{symbol}.csv`.

In [58]:
import calendar


def month_last(mfirst: date) -> date:
    return date(mfirst.year, mfirst.month, calendar.monthrange(mfirst.year, mfirst.month)[1])


def spot_asof(underlying: pd.DataFrame, d: date) -> float:
    sub = underlying.loc[underlying.index <= d]
    if sub.empty:
        return np.nan
    return float(sub["Close"].iloc[-1])


all_rows = []
for month_first in month_starts:
    month_end = month_last(month_first)
    s0 = spot_asof(underlying_df, month_first)
    if not np.isfinite(s0):
        print(f"Skip {month_first}: no underlying spot")
        continue
    atm_call, atm_put = atm_option(symbol, s0, month_first)
    if atm_call is None or atm_put is None:
        print(f"Skip {month_first}: no ATM pair")
        continue

    call_df = option_bars_with_iv(
        atm_call, month_first, month_end, r_by_date, underlying_df, r_fallback
    )
    put_df = option_bars_with_iv(
        atm_put, month_first, month_end, r_by_date, underlying_df, r_fallback
    )

    bdays = pd.bdate_range(month_first, month_end, freq="B")
    panel = pd.DataFrame({"Date": bdays.date})

    u = underlying_df.reset_index().rename(
        columns={"Close": "Stock_Close", "Dividends": "Stock_Dividends"}
    )
    panel = panel.merge(
        u[["Date", "Stock_Close", "Stock_Dividends", "RV"]], on="Date", how="left"
    )
    panel = panel.loc[panel["Stock_Close"].notna()].copy()
    if panel.empty:
        print(f"Skip {month_first}: no rows with Stock_Close (all holidays?)")
        continue

    ts = pd.to_datetime(panel["Date"])
    panel["r"] = r_by_date.reindex(ts).ffill().bfill().fillna(r_fallback).values.astype(float)

    call_df = call_df.rename(
        columns={
            "date": "Date",
            "close": "Call_Close",
            "imp_vol": "Call_imp_vol",
            "delta": "Call_Delta",
            "gamma": "Call_Gamma",
            "vega": "Call_Vega",
            "theta": "Call_Theta",
            "rho": "Call_Rho",
            "vanna": "Call_Vanna",
            "volga": "Call_Volga",
        }
    )
    put_df = put_df.rename(
        columns={
            "date": "Date",
            "close": "Put_Close",
            "imp_vol": "Put_imp_vol",
            "delta": "Put_Delta",
            "gamma": "Put_Gamma",
            "vega": "Put_Vega",
            "theta": "Put_Theta",
            "rho": "Put_Rho",
            "vanna": "Put_Vanna",
            "volga": "Put_Volga",
        }
    )
    call_keep = [
        "Date",
        "Call_Close",
        "Call_imp_vol",
        "Call_Delta",
        "Call_Gamma",
        "Call_Vega",
        "Call_Theta",
        "Call_Rho",
        "Call_Vanna",
        "Call_Volga",
    ]
    put_keep = [
        "Date",
        "Put_Close",
        "Put_imp_vol",
        "Put_Delta",
        "Put_Gamma",
        "Put_Vega",
        "Put_Theta",
        "Put_Rho",
        "Put_Vanna",
        "Put_Volga",
    ]
    for c in call_keep:
        if c not in call_df.columns:
            call_df[c] = np.nan
    for c in put_keep:
        if c not in put_df.columns:
            put_df[c] = np.nan
    panel = panel.merge(call_df[call_keep], on="Date", how="left")
    panel = panel.merge(put_df[put_keep], on="Date", how="left")
    panel["Call_Sym"] = atm_call
    panel["Put_Sym"] = atm_put

    expiry_m, K_m = parse_occ_expiry_strike(atm_call)
    expiry_p, K_p = parse_occ_expiry_strike(atm_put)
    if expiry_m != expiry_p or abs(K_m - K_p) > 1e-6:
        print(f"Warning {month_first}: call/put expiry or strike mismatch {atm_call} vs {atm_put}")
    opt_ff = ["Call_Close", "Call_imp_vol", "Put_Close", "Put_imp_vol"]
    panel[opt_ff] = panel[opt_ff].ffill().bfill()
    panel["_ttm"] = panel["Date"].apply(lambda d: max((expiry_m - d).days / 365.25, 1e-12))
    need_c = panel["Call_imp_vol"].isna() & panel["Call_Close"].notna() & panel["Stock_Close"].notna()
    if need_c.any():
        panel.loc[need_c, "Call_imp_vol"] = panel.loc[need_c].apply(
            lambda row: implied_vol_call(
                row["Call_Close"], row["Stock_Close"], K_m, row["_ttm"], row["r"]
            ),
            axis=1,
        )
    need_p = panel["Put_imp_vol"].isna() & panel["Put_Close"].notna() & panel["Stock_Close"].notna()
    if need_p.any():
        panel.loc[need_p, "Put_imp_vol"] = panel.loc[need_p].apply(
            lambda row: implied_vol_put(
                row["Put_Close"], row["Stock_Close"], K_m, row["_ttm"], row["r"]
            ),
            axis=1,
        )
    for prefix, is_call in (("Call", True), ("Put", False)):
        g = vectorized_leg_greeks(
            panel["Stock_Close"].to_numpy(),
            K_m,
            panel["_ttm"].to_numpy(),
            panel["r"].to_numpy(),
            panel[f"{prefix}_imp_vol"].to_numpy(),
            is_call,
        )
        panel[f"{prefix}_Delta"] = g["Delta"]
        panel[f"{prefix}_Gamma"] = g["Gamma"]
        panel[f"{prefix}_Vega"] = g["Vega"]
        panel[f"{prefix}_Theta"] = g["Theta"]
        panel[f"{prefix}_Rho"] = g["Rho"]
        panel[f"{prefix}_Vanna"] = g["Vanna"]
        panel[f"{prefix}_Volga"] = g["Volga"]

    def _straddle_iv_row(row):
        C, P = row["Call_Close"], row["Put_Close"]
        S, r, T = row["Stock_Close"], row["r"], row["_ttm"]
        if not all(np.isfinite(x) for x in (C, P, S, r, T)):
            return np.nan
        if T <= 1e-10 or S <= 0 or K_m <= 0:
            return np.nan
        try:
            return straddle_implied_vol(
                float(C), float(P), float(S), float(K_m), float(T), r=float(r)
            )
        except (ValueError, ZeroDivisionError, OverflowError):
            return np.nan

    panel["Straddle_imp_vol"] = panel.apply(_straddle_iv_row, axis=1)
    for _g in ("Delta", "Gamma", "Vega", "Theta", "Rho", "Vanna", "Volga"):
        panel[f"Straddle_{_g}"] = panel[f"Call_{_g}"] + panel[f"Put_{_g}"]
    panel.drop(columns=["_ttm"], inplace=True)

    panel["Force_Close"] = panel["Date"] == panel["Date"].max()

    out_cols = [
        "Date",
        "Stock_Close",
        "Stock_Dividends",
        "r",
        "RV",
        "Call_Close",
        "Call_Sym",
        "Put_Close",
        "Put_Sym",
        "Call_imp_vol",
        "Put_imp_vol",
        "Straddle_imp_vol",
        "Straddle_Delta",
        "Straddle_Gamma",
        "Straddle_Vega",
        "Straddle_Theta",
        "Straddle_Rho",
        "Straddle_Vanna",
        "Straddle_Volga",
        "Call_Delta",
        "Call_Gamma",
        "Call_Vega",
        "Call_Theta",
        "Call_Rho",
        "Call_Vanna",
        "Call_Volga",
        "Put_Delta",
        "Put_Gamma",
        "Put_Vega",
        "Put_Theta",
        "Put_Rho",
        "Put_Vanna",
        "Put_Volga",
        "Force_Close",
    ]
    all_rows.append(panel[out_cols])

if not all_rows:
    raise RuntimeError("No monthly panels built; check date range and Polygon responses.")
full = pd.concat(all_rows, ignore_index=True)
out_path = f"DataSet/{symbol}.csv"
full.to_csv(out_path, index=False)
print(f"Wrote {len(full)} rows to {out_path}")

No ATM pair found as of 2024-12-01; tried expiries (first week next month): ['2025-01-03', '2025-01-10']
Skip 2024-12-01: no ATM pair
No ATM pair found as of 2025-01-01; tried expiries (first week next month): ['2025-02-07', '2025-02-14']
Skip 2025-01-01: no ATM pair
No ATM pair found as of 2025-02-01; tried expiries (first week next month): ['2025-03-07', '2025-03-14']
Skip 2025-02-01: no ATM pair
No ATM pair found as of 2025-03-01; tried expiries (first week next month): ['2025-04-04', '2025-04-11']
Skip 2025-03-01: no ATM pair


KeyboardInterrupt: 